# Artist EDA — listen-wiseer

Covers:
1. Top artists by track count
2. Artist popularity distribution
3. Artist genre tag breakdown (top 20)
4. Artist popularity vs track count scatter

In [ ]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

import warnings

warnings.filterwarnings("ignore")
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
DB = str(ROOT / "data/listen_wiseer.db")
con = duckdb.connect(DB, read_only=True)

## 1. Top artists by track count

In [ ]:
top_artists = con.execute("""
    SELECT a.artist_id, a.popularity, a.genres,
           COUNT(ta.track_id) AS track_count
    FROM track_artists ta
    JOIN artists a USING (artist_id)
    GROUP BY a.artist_id, a.popularity, a.genres
    ORDER BY track_count DESC
    LIMIT 20
""").df()
top_artists

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_artists["artist_id"], top_artists["track_count"])
ax.set_title("Top 20 artists by track count")
ax.set_xlabel("track count")
ax.invert_yaxis()
plt.tight_layout()

## 2. Artist popularity distribution

In [ ]:
artist_pop = con.execute("SELECT popularity FROM artists WHERE popularity IS NOT NULL").df()
fig, ax = plt.subplots(figsize=(7, 3))
sns.histplot(artist_pop["popularity"], bins=30, kde=True, ax=ax)
ax.set_title("Artist popularity distribution")
plt.tight_layout()

## 3. Artist genre tag breakdown

In [ ]:
import ast


def parse_genres(s):
    if not isinstance(s, str) or not s.strip():
        return []
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return [g.strip() for g in s.strip("[]").replace("'", "").split(",") if g.strip()]


artists_df = con.execute("SELECT artist_id, genres, popularity FROM artists").df()
artists_df["genre_list"] = artists_df["genres"].apply(parse_genres)
genre_series = artists_df.explode("genre_list")["genre_list"].dropna()
genre_series = genre_series[genre_series != ""]
top_tags = genre_series.value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top_tags.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_title("Top 20 artist genre tags")
ax.set_xlabel("artist count")
plt.tight_layout()

## 4. Artist popularity vs track count

In [ ]:
artist_track_counts = con.execute("""
    SELECT a.artist_id, a.popularity, COUNT(ta.track_id) AS track_count
    FROM track_artists ta
    JOIN artists a USING (artist_id)
    GROUP BY a.artist_id, a.popularity
""").df()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(artist_track_counts["popularity"], artist_track_counts["track_count"], alpha=0.5, s=20)
ax.set_xlabel("artist popularity")
ax.set_ylabel("track count in library")
ax.set_title("Artist popularity vs track count")
plt.tight_layout()

In [ ]:
con.close()
print("Done.")